# Customer Review Sentiment — DistilBERT Walkthrough

**Goal of this notebook:** Walk through every step of using a pre-trained DistilBERT model on customer reviews.  
Run one cell at a time. After each cell, look at the output, change something if you want, then continue.

This matches the tools from this week’s class (Transformers, pipelines, tokens, pre-trained models).


## Step 1: Load the sample data

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("sample_customer_reviews.csv")
print("Shape:", df.shape)
print("\nFirst 10 rows:")
df.head(10)


### What to look at
- How many rows and columns?
- Are there any empty cells?
- What do the sentiment labels look like?

**Try this:** Change `df.head(10)` to `df.head(20)` and re-run the cell.


## Step 2: Inspect and repair the data

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nUnique sentiments:")
print(df["sentiment"].unique())

print("\nExample of extra whitespace (row 6 if it exists):")
print(repr(df.loc[5, "review_text"]) if len(df) > 5 else "N/A")


In [ ]:
# Repair steps
df = df.dropna(subset=["review_text"]).copy()
df["review_text"] = df["review_text"].astype(str).str.strip()
df["sentiment"] = df["sentiment"].str.strip().str.title()

print("After cleaning:")
print(df["sentiment"].value_counts())
df[["review_id", "review_text", "sentiment"]].head(8)


**Try this:** Add a new line before the cleaning and print one review with extra spaces so you can see the difference.


## Step 3: Tokenization — what the model actually sees

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

sample = df.iloc[0]["review_text"]
print("Original:", sample)

tokens = tokenizer.tokenize(sample)
print("\nTokens:", tokens)
print("Number of tokens:", len(tokens))

ids = tokenizer.encode(sample)
print("\nToken IDs:", ids)
print("Decoded:", tokenizer.decode(ids))


**Try this:** Change `df.iloc[0]` to `df.iloc[5]` or pick a longer review and re-run to see different tokens.


## Step 4: Load the pre-trained DistilBERT pipeline

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=128
)
print("Pipeline loaded.")
print("Model only knows POSITIVE and NEGATIVE.")


## Step 5: Run predictions on a few reviews first

In [ ]:
# Test on just the first 5 reviews so you can see the results clearly
sample_reviews = df["review_text"].head(5).tolist()
results = classifier(sample_reviews)

for text, result in zip(sample_reviews, results):
    print(f"{text}")
    print(f"  → {result['label']} ({result['score']:.3f})\n")


**Try this:** Add your own review to the list and re-run the cell.


## Step 6: Run on the whole dataset and look at results

In [ ]:
results = classifier(df["review_text"].tolist())
df["predicted_label"] = [r["label"] for r in results]
df["confidence"] = [round(r["score"], 4) for r in results]

df[["review_text", "sentiment", "predicted_label", "confidence"]].head(15)


## Step 7: Evaluate only on Positive / Negative labels

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

binary = df[df["sentiment"].isin(["Positive", "Negative"])].copy()
binary["true_label"] = binary["sentiment"].map({"Positive": "POSITIVE", "Negative": "NEGATIVE"})

print("Accuracy:", round(accuracy_score(binary["true_label"], binary["predicted_label"]), 3))
print()
print(classification_report(binary["true_label"], binary["predicted_label"]))


## Step 8: Experiment — change a review and re-predict

In [ ]:
# Change this sentence and re-run the cell
my_review = "The product is okay but shipping was slow."

result = classifier(my_review)[0]
print(my_review)
print(f"→ {result['label']} ({result['score']:.3f})")


## How the pieces fit together

1. Raw text → Tokenizer turns words into token IDs  
2. Token IDs + positions → Transformer encoder layers  
3. Self-attention lets every token look at every other token  
4. Final [CLS] representation → small classification head  
5. Softmax → probability for POSITIVE vs NEGATIVE  

You did not train the model. It was already pre-trained and fine-tuned. That is why a few lines of code can do the work that used to require building TF-IDF features yourself.
